In [3]:
from countryinfo import CountryInfo

In [ ]:
Curaçao

In [1]:
import json
capitals_cache = {}
with open("../data/cache/capitals_cache.json") as f:
    capitals_cache = json.load(f)

In [4]:
def get_capital(country):
    if country in capitals_cache:
        return capitals_cache[country]
    
    try:
        capital = CountryInfo(country).capital()
        capitals_cache[country] = capital
        return capital
    except Exception as e:
        raise e

In [5]:
get_capital("Curaçao")

'Willemstad'

In [ ]:
import time
import json
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderRateLimited

geolocator = Nominatim(
    user_agent="nations-matches-prediction",
    timeout=10
)

CACHE_FILE = "coords_cache.json"

# Load existing cache so you never re-geocode the same place
try:
    with open(CACHE_FILE) as f:
        coords = json.load(f)
except (FileNotFoundError, json.JSONDecodeError):
    coords = {}

def get_coords(place):
    if place in coords:
        return coords[place]  # skip already-geocoded places
    
    for attempt in range(5):
        try:
            time.sleep(1.2)  # always wait before each request
            location = geolocator.geocode(place)
            result = (location.latitude, location.longitude) if location else None
            coords[place] = result

            # Save after every successful call
            with open(CACHE_FILE, "w") as f:
                json.dump(coords, f)

            return result

        except GeocoderRateLimited:
            wait = 60
            print(f"Rate limited on '{place}', waiting {wait}s (attempt {attempt+1}/5)")
            time.sleep(wait)

    print(f"Giving up on '{place}' after 5 attempts")
    coords[place] = None
    return None

# take input "city, country"